# Improving earthquake metadata for GNSS time series analysis using Okada-based co-seismic deformation modelling
# ***Verification of Finite Source Model with IGS***
*** 
***

## **1. Scaling Laws**

The IGS Network follows the following convention for scaling laws, based on Métivier et al.'s work [1].

For earthquakes with magnitudes $<7.6$, the width, length, and slip of the rupture are estimated from Yen & Ma [2]. Thus, for earthquakes with magnitude $<7.27$ or scalar moment $\leq 10^{20} [N-m]$:

\begin{cases}
\log L = \frac{1}{2}\log M_0-8.08 [km]\\
\log W = \frac{1}{2}\log M_0-8.08 [km]\\
\tag{1}
\end{cases}

We will define slip as:

\begin{equation}
U = \frac{M_0}{\mu\cdot L\cdot W} [km]
\tag{2}
\end{equation}

For earthquakes with magnitude $>7.27$ or scalar moment $> 10^{20} [N-m]$, Yen & Ma [2] use the following formulae instead:

\begin{cases}
\log L = \frac{1}{3}\log M_0-4.84 [km]\\
\log W = \frac{1}{3}\log M_0-5.27 [km]\\
\log U = \frac{1}{3}\log M_0-4.37 [cm]\\
\tag{3}
\end{cases}

For earthquakes with magnitudes $\geq7.6$, the width, length, and slip of the rupture are estimated from Mai & Beroza [3]. For strike-slip:

\begin{cases}
\log L = 0.40\log M_0-6.31 [km]\\
\log W = 0.17\log M_0-2.18 [km]\\
\log U = 0.43\log M_0-6.03 [cm]
\tag{4}
\end{cases}

For all other focal mechanisms:

\begin{cases}
\log L = 0.40\log M_0-6.39 [km]\\
\log W = 0.35\log M_0-5.51 [km]\\
\log U = 0.25\log M_0-2.62 [cm]
\tag{5}
\end{cases}

***

[1] L. Métivier, X. Collilieux, D. Lercier, Z. Altamimi, and F. Beauducel, “Global coseismic deformations, GNSS time series analysis, and Earthquake Scaling Laws,” Journal of Geophysical Research: Solid Earth, vol. 119, no. 12, pp. 9095–9109, Dec. 2014. doi:10.1002/2014jb011280

[2] Y.-T. Yen and K.-F. Ma, “Source-scaling relationship for M 4.6-8.9 earthquakes, specifically for earthquakes in the collision zone of Taiwan,” Bulletin of the Seismological Society of America, vol. 101, no. 2, pp. 464–481, Mar. 2011. doi:10.1785/0120100046

[3] P. M. Mai and G. C. Beroza, “Source scaling properties from finite-fault-rupture models,” Bulletin of the Seismological Society of America, vol. 90, no. 3, pp. 604–615, Jun. 2000. doi:10.1785/0119990126 

## **2. Okada and Scaling Laws Functions**

In [1]:
import numpy as np

### 2.1. Okada's Coordinate System

In [2]:
def coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, U, W):
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, stalat, stalon)
    dEast = distance_m*np.sin(azimuth*np.pi/180.0) #m
    dNorth = distance_m*np.cos(azimuth*np.pi/180.0) #m

    x = dEast * np.sin(strike*np.pi/180.0) + dNorth * np.cos(strike*np.pi/180.0) #m
    y = -dEast * np.cos(strike*np.pi/180.0) + dNorth * np.sin(strike*np.pi/180.0) #m

    d = abs(evdep) #m
    delta = dip*np.pi/180.0 #radians
    
    p = y*np.cos(delta)+d*np.sin(delta) #m
    q = y*np.sin(delta)-d*np.cos(delta) #m

    mu = 10**9*mu #convert GPa to N/m^2
    lam = 10**9*lam #convert GPa to N/m^2
    return(
        U*np.cos(rake*np.pi/180.0), #m #U1
        U*np.sin(rake*np.pi/180.0), #m #U2
        x, #m #x
        y, #m #y
        d, #m #d
        mu, #GPa->N/m^2 #mu
        lam, #GPa->N/m^2 #lam
        delta, #rad #delta
        p, #m #p
        q #m #q
    )

### 2.2. Parameters needed for Chinnery Notation

In [3]:
def chinnery (x, p, q, L, W, delta):
        xi = [x+L/2,x-L/2] #m #xi
        eta = [p+W/2,p-W/2] #m #eta

        ytil = [0,0]
        dtil = [0,0]
        R = [[0,0],[0,0]]
        X = [0,0]

        for i in range(2):
            ytil[i]=eta[i]*np.cos(delta)+q*np.sin(delta) #m #ytil
            dtil[i]=eta[i]*np.sin(delta)-q*np.cos(delta) #m #dtil
            X[i]=np.sqrt(xi[i]**2+q**2) #m #X
            for j in range(2):
                R[i][j]=np.sqrt(xi[i]**2.0+eta[j]**2.0+q**2.0) #m #R

        return(xi, eta, ytil, dtil, R, X)

### 2.3. Displacement Functions and Intermediate Variables

In [4]:
def disp_strike (xi, eta, q, ytil, dtil, R, I1, I2, I4, delta, U1):
    if(abs(q)<1e-14):
        atanterm=0
    else:
        atanterm=np.arctan2((xi*eta),(q*R))
    if (abs(R+eta)<1e-14):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
    -U1/(2.0*np.pi)*(((xi*q)/R*Retaterm)+atanterm+I1*np.sin(delta)),
    -U1/(2.0*np.pi)*(((ytil*q)/R*Retaterm)+(q*np.cos(delta))*Retaterm+I2*np.sin(delta)),
    -U1/(2.0*np.pi)*(((dtil*q)/R*Retaterm)+(q*np.sin(delta))*Retaterm+I4*np.sin(delta))
    )

In [5]:
def disp_dip (xi, eta, q, ytil, dtil, R, I1, I3, I5, delta, U2):
    if(abs(q)<1e-14):
        atanterm=0
    else:
        atanterm=np.arctan2((xi*eta),(q*R))
    return (
    -U2/(2.0*np.pi)*(((q/R)-I3*np.sin(delta)*np.cos(delta))),
    -U2/(2.0*np.pi)*(((ytil*q)/(R*(R+xi))+np.cos(delta)*atanterm-I1*np.sin(delta)*np.cos(delta))),
    -U2/(2.0*np.pi)*(((dtil*q)/(R*(R+xi))+np.sin(delta)*atanterm-I5*np.sin(delta)*np.cos(delta)))
    )

In [6]:
def disp_I (xi, eta, q, ytil, dtil, R, X, lam, mu, delta):
    if (abs(R+eta)<1e-14):
        lnterm = -np.log(R-eta)
    else:
        lnterm = np.log(R+eta)
    if (abs(np.cos(delta))<1e-14):
        I3 = mu/(2*(lam+mu))*(eta/(R+dtil)+(ytil*q)/(R+dtil)**2-lnterm)
        I2 =mu/(lam+mu)*(-lnterm)-I3
        return(
            -mu/(2*(lam+mu))*xi*q/(R+dtil)**2,
            I2,
            I3,
            mu/(lam+mu)*q/(R+dtil),
            mu/(lam+mu)*xi*np.sin(delta)/(R+dtil)
        )
    else:
        I4 = mu/(lam+mu)/np.cos(delta)*(np.log(R+dtil)-np.sin(delta)*lnterm)
        I3 = mu/(lam+mu)*(1/np.cos(delta)*ytil/(R+dtil)-lnterm)+np.sin(delta)/np.cos(delta)*I4
        I5 = 0 if abs(xi)<1e-14 else mu/(lam+mu)*2.0/np.cos(delta)*np.arctan2((eta*(X+q*np.cos(delta))+X*(R+X)*np.sin(delta)),(xi*(R+X)*np.cos(delta)))
        I2 =mu/(lam+mu)*(-lnterm)-I3
        return(
            mu/(lam+mu)*(-1)/np.cos(delta)*xi/(R+dtil)-np.sin(delta)/(np.cos(delta))*I5,
            I2,
            I3,
            I4, 
            I5
        )

In [7]:
def displacement (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, U, L, W) :
    U1, U2, x, y, d, mu, lam, delta, p, q = coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, U, W)
    xi, eta, ytil, dtil, R, X = chinnery (x, p, q, L, W, delta)
    ux, uy, uz = [], [], []
    for i in range(2):
        for j in range (2):
            I1, I2, I3, I4, I5 = disp_I (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], X[i], lam, mu, delta)
            ux_strike, uy_strike, uz_strike = disp_strike (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I2, I4, delta, U1)
            ux_dip, uy_dip, uz_dip = disp_dip (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I3, I5, delta, U2)
            ux.append(ux_strike + ux_dip)
            uy.append(uy_strike + uy_dip)
            uz.append(uz_strike + uz_dip)
    return ( #chinnery
        ux[0]-ux[1]-ux[2]+ux[3], #ux
        uy[0]-uy[1]-uy[2]+uy[3], #uy
        uz[0]-uz[1]-uz[2]+uz[3] #uz
    )

### 2.4. Scaling Laws

In [36]:
def scaling (mag, m0, mu, rake):
    mu = 10**9*mu #convert GPa to N/m^2
    if (mag<7.6): #Yen & Ma
        if (m0<=10.0**20.0): #N-m
            L = 10.0**(0.5*np.log10(m0)-8.08)*1000.0 #km -> m
            W = 10.0**(0.5*np.log10(m0)-8.08)*1000.0 #km -> m
            U = m0/mu/L/W #m
        else:
            L = 10.0**(1.0/3.0*np.log10(m0)-4.84)*1000.0 #km -> m
            W = 10.0**(1.0/3.0*np.log10(m0)-5.27)*1000.0 #km -> m
            U = 10.0**(1.0/3.0*np.log10(m0)-4.37)/100.0 #cm -> m
    else: #Mai & Beroza
        if abs(rake) <= 30 or abs(abs(rake) - 180) <= 30: #strike-slip
            L = 10.0**(0.40*np.log10(m0)-6.31)*1000.0 #km -> m
            W = 10.0**(0.17*np.log10(m0)-2.18)*1000.0 #km -> m
            U = 10.0**(0.43*np.log10(m0)-6.03)/100.0 #cm -> m
        else:
            L = 10.0**(0.40*np.log10(m0)-6.39)*1000.0 #km -> m
            W = 10.0**(0.35*np.log10(m0)-5.51)*1000.0 #km -> m
            U = 10.0**(0.25*np.log10(m0)-2.62)/100.0 #cm -> m
    return(L, W, U)

### 2.5. Rotation to LonLat Coordinates

In [9]:
def NE_rotate(ux, uy, uz, strike):
    return(
        (ux * np.cos(strike*np.pi/180.0) + uy * np.sin(strike*np.pi/180.0))*10.0**3.0, #mm #dN
        (ux * np.sin(strike*np.pi/180.0) - uy * np.cos(strike*np.pi/180.0))*10.0**3.0, #mm #dE
        uz*10.0**3.0 #mm #dH
    )

## **3. Earthquakes**

*Note: Run only the cell of the Earthquake and Station prameters of the Earthquake you wanna generate, then proceed to Section 4 of this notebook.*

In [10]:
from obspy.geodetics import gps2dist_azimuth
import pandas as pd

## **3.1. Yen & Ma, Equation 1, M$\leq$7.27**

### *3.1.1. El Salvador, 6.2*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008002.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=1&day=5&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=6.2&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [11]:
# Earthquake Parameters

location = 'ElSalvador'
event_time = "2025-01-05 17:18:48"

strike1, dip1, rake1 = 174.0, 9.0, -46.0 #Nodal Plane 1
strike2, dip2, rake2 = 309.0, 84.0, -96.0 #Nodal Plane 2

evlat = 13.050 #degN
evlon = -89.170 #degE
evdep = 62.3*1000 #km to m

mag = 6.2
m0 = 2.21e+25*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['SSIA']
sta_lats = [13.697] #degN
sta_lons = [-89.116] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

1
1
1


In [12]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.1.2. New Guinea, 6.7*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008191.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=10&day=7&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=6.7&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [13]:
# Earthquake Parameters

location = 'NewGuinea'
event_time = "2025-10-07 11:05:18"

strike1, dip1, rake1 = 331.0, 35.0, -37.0 #Nodal Plane 1
strike2, dip2, rake2 = 92.0, 70.0, -119.0 #Nodal Plane 2

evlat = -6.780 #degN
evlon = 146.770 #degE
evdep = 106.4*1000 #km to m

mag = 6.7
m0 = 1.31e+26*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['LAE1']
sta_lats = [-6.674] #degN
sta_lons = [146.993] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

1
1
1


In [14]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.1.3. Japan, 6.8*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008006.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=1&day=13&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=6.8&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [15]:
# Earthquake Parameters

location = 'Japan'
event_time = "2025-01-13 12:19:32"

strike1, dip1, rake1 = 208.0, 24.0, 85.0 #Nodal Plane 1
strike2, dip2, rake2 = 33.0, 66.0, 92.0 #Nodal Plane 2

evlat = 31.810 #degN
evlon = 131.570 #degE
evdep = 34.2*1000 #km to m

mag = 6.8
m0 = 1.82e+26*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['AIRA']
sta_lats = [31.824] #degN
sta_lons = [130.600] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

1
1
1


In [16]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.1.4. Xizang, 7.1*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008003.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=1&day=7&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.1&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [17]:
# Earthquake Parameters

location = 'Xizang'
event_time = "2025-01-07 01:05:17"

strike1, dip1, rake1 = 358.0, 43.0, -85.0 #Nodal Plane 1
strike2, dip2, rake2 = 171.0, 47.0, -95.0 #Nodal Plane 2

evlat = 28.640 #degN
evlon = 87.360 #degE
evdep = 12.0*1000 #km to m

mag = 7.1
m0 = 5.28e+26*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['LHAZ']
sta_lats = [29.657] #degN
sta_lons = [91.104] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

1
1
1


In [18]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

## **3.2. Yen & Ma, Equation 2, 7.27<M<7.6**

### *3.2.1. Alaska M7.3*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008115.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=7&day=16&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.3&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [19]:
# Earthquake Parameters

location = 'Alaska'
event_time = "2025-07-16 20:37:39"

strike1, dip1, rake1 = 345.0, 58.0, 157.0 #Nodal Plane 1
strike2, dip2, rake2 = 88.0, 71.0, 34.0 #Nodal Plane 2

evlat = 54.550 #degN
evlon = -160.470 #degE
evdep = 37.4*1000 #km to m

mag = 7.3
m0 = 9.48e+26*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['AC24']
sta_lats = [58.682] #degN
sta_lons = [-156.653] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

1
1
1


In [20]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.2.2. Philippines M7.4*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008192.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=10&day=10&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.4&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [21]:
# Earthquake Parameters

location = 'Philippines'
event_time = "2025-10-10 01:44:00"

strike1, dip1, rake1 = 175.0, 40.0, 64.0 #Nodal Plane 1
strike2, dip2, rake2 = 27.0, 54.0, 110.0 #Nodal Plane 2

evlat = 7.260 #degN
evlon = 126.750 #degE
evdep = 44.4*1000 #km to m

mag = 7.4
m0 = 1.67e+27*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['PGEN', 'PPPC']
sta_lats = [6.065, 9.773] #degN
sta_lons = [125.132, 118.740] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

2
2
2


In [22]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.2.3. Drake Passage M7.5*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008073.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=5&day=2&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.5&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [23]:
# Earthquake Parameters

location = 'DrakePassage'
event_time = "2025-05-02 12:58:26"

strike1, dip1, rake1 = 326.0, 37.0, 100.0 #Nodal Plane 1
strike2, dip2, rake2 = 133.0, 53.0, 82.0 #Nodal Plane 2

evlat = -56.780 #degN
evlon = -68.210 #degE
evdep = 18.8*1000 #km to m

mag = 7.5
m0 = 2.18e+27*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['FALK', 'PARC', 'RGDG', 'RIO2']
sta_lats = [-51.694, -53.137, -53.786, -53.785] #degN
sta_lons = [-57.874, -70.880, -67.752, -67.751] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

4
4
4


In [24]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

## **3.3. Mai & Beroza, Equation 3, M$\geq$7.6**

### *3.3.1. Honduras M7.6*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008023.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=2&day=8&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.6&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [25]:
# Earthquake Parameters

location = 'Honduras'
event_time = "2025-02-08 23:23:14"

strike1, dip1, rake1 = 79.0, 81.0, 7.0 #Nodal Plane 1
strike2, dip2, rake2 = 348.0, 83.0, 171.0 #Nodal Plane 2

evlat = 17.700 #degN
evlon = -82.460 #degE
evdep = 18.8*1000 #km to m

mag = 7.6
m0 = 2.87e+27*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['GUAT', 'MANA', 'SCUB', 'SSIA']
sta_lats = [14.590, 12.149, 20.012, 13.697] #degN
sta_lons = [-90.520, -86.249, -75.762, -89.116] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

4
4
4


In [26]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.3.2. Myanmar M7.7*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008044.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=3&day=28&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.7&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [27]:
# Earthquake Parameters

location = 'Myanmar'
event_time = "2025-03-28 06:20:54"

strike1, dip1, rake1 = 353.0, 60.0, 175.0 #Nodal Plane 1
strike2, dip2, rake2 = 85.0, 86.0, 30.0 #Nodal Plane 2

evlat = 22.010 #degN
evlon = 95.920 #degE
evdep = 20.1*1000 #km to m

mag = 7.7
m0 = 5.13e+27*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['BNEU', 'CMUM', 'CUSV', 'CUUT', 'LHAZ', 'NKAY', 'PBR4', 'SHLG', 'TOAY']
sta_lats = [21.644, 18.761, 13.736, 13.736, 29.657, 17.719, 11.637, 25.674, 16.073] #degN
sta_lons = [101.917, 98.932, 100.534, 100.534, 91.104, 105.153, 92.712, 91.913, 106.627] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

9
9
9


In [28]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.3.3. Turkey M7.8*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2023/007685.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2023&mo=02&day=6&oyr=1976&omo=1&oday=1&jyr=&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.8&umw=7.8&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0


In [29]:
# Earthquake Parameters

location = 'Turkey'
event_time = "2023-02-06 01:17:36"

strike1, dip1, rake1 = 51.0, 70.0, -4.0 #Nodal Plane 1
strike2, dip2, rake2 = 143.0, 86.0, -160.0 #Nodal Plane 2

evlat = 37.170 #degN
evlon = 37.030 #degE
evdep = 15.1*1000 #km to m

mag = 7.8
m0 = 5.8e+27*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['ANK2', 'ARUC', 'BSHM', 'DRAG', 'DYNG', 'HAMD', 'ISBA', 'ISTA', 'IZMI', 'KHAR', 'KRS1', 'MERS', 'MIKL', 'NICO', 'ORID', 'POLV', 'RAMO', 'SOFI', 'TEHN', 'TUBI', 'ZECK']
sta_lats = [39.843, 40.286, 32.779, 31.593, 38.079, 34.869, 33.341, 41.104, 38.395, 50.005, 40.588, 36.566, 46.973, 35.141, 41.127, 49.603, 30.598, 42.556, 35.697, 40.787, 43.788] #degN
sta_lons = [32.775, 44.086, 35.020, 35.392, 23.932, 48.534, 44.438, 29.019, 27.082, 36.239, 43.093, 34.256, 31.973, 33.396, 20.794, 34.543, 34.763, 23.395, 51.334, 29.451, 41.565] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

21
21
21


In [30]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.3.4. Philippines M7.8*
> IGS Solution Parameters: \
https://lists.igs.org/pipermail/igsstation/2026/008316.html

> Source: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2026&mo=06&day=7&oyr=1976&omo=1&oday=1&jyr=&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.8&umw=7.8&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0


In [31]:
# Earthquake Parameters

location = 'Philippines'
event_time = "2026-06-07 23:37:42"

strike1, dip1, rake1 = 165.0, 44.0, 84.0 #Nodal Plane 1
strike2, dip2, rake2 = 353.0, 46.0, 96.0 #Nodal Plane 2

evlat = 5.590 #degN
evlon = 125.050 #degE
evdep = 44.0*1000 #km to m

mag = 7.8
m0 = 6.13e+27*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

sta_ids = ['BTNG', 'PGEN', 'PPPC']
sta_lats = [1.439, 6.065, 9.773] #degN
sta_lons = [125.190, 125.132, 118.740] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

3
3
3


In [32]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

### *3.3.5. Kamcha M8.7*
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2025/008126.html

> Source: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2025&mo=7&day=29&oyr=1976&omo=1&oday=1&jyr=1976&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=8.7&umw=10&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

In [33]:
# Earthquake Parameters

location = 'Kamcha'
event_time = "2025-07-29 23:24:51"

strike1, dip1, rake1 = 214.0, 18.0, 84.0 #Nodal Plane 1
strike2, dip2, rake2 = 40.0, 72.0, 92.0 #Nodal Plane 2

evlat = 52.510 #degN
evlon = 160.260 #degE
evdep = 36.3*1000 #km to m

mag = 8.7
m0 = 1.58e+29*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

# Station Parameters

# Station Parameters

sta_ids = [
    'BADG', 'BJFS', 'BJNM', 'CHAN', 'CHOF',
    'IRKJ', 'IRKM', 'ISHI', 'MAG0', 'MCIL',
    'MIZU', 'MSSA', 'MTKA', 'NRIL', 'NVSK',
    'OSN3', 'OSN4', 'PETS', 'SMST', 'STK2',
    'SUWN', 'TIXI', 'TKBG', 'TSK2', 'TSKB',
    'ULAB', 'USUD', 'YAKT', 'YONS', 'YSSK'
]

sta_lats = [
    51.770, 39.609, 40.245, 43.790, 35.675,
    52.219, 52.219, 36.209, 59.576, 24.290,
    39.135, 36.140, 35.680, 69.362, 54.841,
    37.083, 37.083, 53.023, 33.578, 43.529,
    37.276, 71.634, 36.068, 36.106, 36.106,
    47.865, 36.133, 62.031, 37.541, 47.030
]  # deg N

sta_lons = [
    102.235, 115.892, 116.224, 125.443, 139.531,
    104.316, 104.316, 140.219, 150.770, 153.979,
    141.133, 138.352, 139.561, 88.360, 83.235,
    127.034, 127.034, 158.650, 135.937, 141.845,
    127.054, 128.866, 140.131, 140.087, 140.088,
    107.052, 138.362, 129.680, 127.001, 142.717
]  # deg E

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

30
30
30


In [34]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)

## **4. Calculating Surface Deformation**

*Note: Run after you have run the cell corresponsing to the earthquake you want to examine.*

In [35]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

for i in range(len(sta_ids)): # For every station
    
    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake1)
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    #Compute the slip (U), length (L), and width of rupture (W) for Nodal Plane 1
    L, W, U = scaling (mag, m0, mu, rake2)

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/{location}_M{mag}_{event_time}.csv", index=False)